# 4. Data Cleaning & Preprocessing <a id="4-data-cleaning" name="4-data-cleaning"></a>



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CLEANING PIPELINE
# ═══════════════════════════════════════════════════════════════════

df = df_raw.copy()
steps = {}


# Step 1 : Parse dates
df["release_date"]    = pd.to_datetime(df["release_date"], errors="coerce") # transform invalid dates into NaT
df["release_year"]    = df["release_date"].dt.year.astype("Int64")
df["release_month"]   = df["release_date"].dt.month.astype("Int64")
df["release_quarter"] = df["release_date"].dt.quarter.astype("Int64")
df["release_decade"]  = (df["release_year"] // 10 * 10).astype("Int64") # calcule dacade
steps["After date parsing"] = len(df)


# Step 2 numeric_types: replace missing values with NaN
# and then replace NaN values with 0
# this step is important because we have columns with mixed data types
for col in ["runtime","budget","revenue","vote_count","vote_average","popularity"]:
    df[col]=pd.to_numeric(df[col],errors="coerce").fillna(0)


# Step 3: Remove duplicates
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
steps["After deduplication"] = len(df)


# Step 4: Keep Released only
df = df[df["status"] == "Released"].copy()
steps["After status=Released filter"] = len(df)

# Step 5: Remove adult content
df = df[df["adult"] == False].copy()
steps["After adult=False filter"] = len(df)

# Step 6: Runtime filter (45–300 min)
df = df[df["runtime"].between(45, 300)].copy()
steps["After runtime (45-300 min)"] = len(df)

# Step 7: Minimum votes (data quality)
df = df[df["vote_count"] >= 10].copy()
steps["After min 10 votes"] = len(df)

# Step 8: Genres not null exclude
df = df[df["genres"].notna() & (df["genres"].str.strip() != "")].copy()
steps["After genres not null"] = len(df)

df = df.reset_index(drop=True)
steps["FINAL"] = len(df)

print("CLEANING PIPELINE SUMMARY:")
for step, count in steps.items():
    removed = list(steps.values())[list(steps.keys()).index(step)-1] - count if list(steps.keys()).index(step) > 0 else 0
    print(f"  {step:<40} → {count:,} rows  (removed: {removed})")

CLEANING PIPELINE SUMMARY:
  After date parsing                       → 5,500 rows  (removed: 0)
  After deduplication                      → 5,500 rows  (removed: 0)
  After status=Released filter             → 5,477 rows  (removed: 23)
  After adult=False filter                 → 5,477 rows  (removed: 0)
  After runtime (45-300 min)               → 5,378 rows  (removed: 99)
  After min 10 votes                       → 5,261 rows  (removed: 117)
  After genres not null                    → 5,261 rows  (removed: 0)
  FINAL                                    → 5,261 rows  (removed: 0)


In [ ]:
# Derived columns
# Primary genre
df["primary_genre"] = df["genres"].apply(
    lambda x: str(x).split(",")[0].strip() if pd.notna(x) and str(x).strip() else "Unknown")
df["n_genres"] = df["genres"].apply(
    lambda x: len([g for g in str(x).split(",") if g.strip()]) if pd.notna(x) else 0)

# Financial
df["budget_known"]  = df["budget"]  > 100_000
df["revenue_known"] = df["revenue"] > 100_000
df["profit"] = np.where(df["budget_known"] & df["revenue_known"],
                         df["revenue"] - df["budget"], np.nan)
df["roi"]    = np.where(df["budget_known"] & df["revenue_known"] & (df["budget"] > 0),
                         (df["revenue"] - df["budget"]) / df["budget"], np.nan)

# IMDb weighted rating: WR = (v/(v+m))*R + (m/(v+m))*C # The official IMDb formula for ranking movies (Weighted Rating)
m = df["vote_count"].quantile(0.75) # m : Minimum vote count required / v : Number of votes for the movie
C = df["vote_average"].mean() # C : Global mean rating across all movies / R : Mean rating for the movie
df["weighted_rating"] = ((df["vote_count"]/(df["vote_count"]+m))*df["vote_average"] +
                          (m/(df["vote_count"]+m))*C).round(3)

# WR = movie rating weight + global mean weight
# A movie with few votes is pulled toward the global mean: a movie with vote_count = 5 and vote_average = 9.5
# is not reliable > WR will be close to "C". A movie with vote_count = 50000 and vote_average = 9.0 > highly reliable > WR close to 9.0

# Categorical tiers
df["rating_tier"] = pd.cut(df["vote_average"], bins=[0,4,6,7,8,10],
    labels=["Poor (<4)","Fair (4-6)","Good (6-7)","Great (7-8)","Excellent (8+)"])
df["runtime_category"] = df["runtime"].apply(
    lambda r: "Short (<80min)" if r<80 else
             ("Standard (80-110min)" if r<110 else
             ("Long (110-140min)" if r<140 else "Epic (140+min)")))
df["budget_tier"] = pd.cut(df["budget"],
    bins=[-1,0,5_000_000,30_000_000,100_000_000,float("inf")],
    labels=["Unknown","Low Budget","Mid Budget","High Budget","Blockbuster"])
try:
    df["popularity_tier"] = pd.qcut(df["popularity"], q=4,
        labels=["Low","Medium","High","Very High"], duplicates="drop")
except:
    df["popularity_tier"] = "Medium"

print(" Derived columns added:")
new_cols = ["release_year","release_decade","primary_genre","n_genres",
            "profit","roi","weighted_rating","rating_tier","runtime_category",
            "budget_tier","popularity_tier"]

for c in new_cols:
    print(f"  + {c}")
print(f"\nFinal dataset: {df.shape}")
display(df.head(3))

 Derived columns added:
  + release_year
  + release_decade
  + primary_genre
  + n_genres
  + profit
  + roi
  + weighted_rating
  + rating_tier
  + runtime_category
  + budget_tier
  + popularity_tier

Final dataset: (5261, 43)


,id,imdb_id,title,original_title,original_language,overview,tagline,genres,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,...,release_year,release_month,release_quarter,release_decade,primary_genre,n_genres,budget_known,revenue_known,profit,roi,weighted_rating,rating_tier,runtime_category,budget_tier,popularity_tier
0,11,tt0076759,Star Wars,Star Wars,en,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...","Adventure,Action,Science Fiction",1977-05-25,121,11000000,775398007,22.71,8.20,22179,...,1977,5,2,1970,Adventure,3,True,True,764398007.00,69.49,8.03,Excellent (8+),Long (110-140min),Mid Budget,Very High
1,12,tt0266543,Finding Nemo,Finding Nemo,en,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,"Animation,Family,Adventure",2003-05-30,100,94000000,940335536,18.92,7.82,20443,...,2003,5,2,2000,Animation,3,True,True,846335536.00,9.00,7.69,Great (7-8),Standard (80-110min),High Budget,Very High
2,13,tt0109830,Forrest Gump,Forrest Gump,en,A man with a low IQ has accomplished great thi...,The world will never be the same once you've s...,"Comedy,Drama,Romance",1994-06-23,142,55000000,677387716,26.63,8.46,29561,...,1994,6,2,1990,Comedy,3,True,True,622387716.00,11.32,8.30,Excellent (8+),Epic (140+min),High Budget,Very High
